# Chapter 1 — The photocentre wobble: how a companion perturbs Gaia astrometry

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/gaia-nss-tutorial/blob/main/notebooks/01_photocentre_wobble.ipynb)

**Goal.** Gaia does not see most binaries as two stars: the two images merge into one, and Gaia measures the position
of their combined centre of light, the **photocentre**. As the stars orbit each other, the photocentre traces a small
ellipse on the sky. A single-star model (position, parallax, proper motion) cannot follow it, and the misfit shows up
as an inflated $\chi^2$, i.e. a large RUWE and astrometric excess noise (Belokurov et al. 2020,
https://ui.adsabs.harvard.edu/abs/2020MNRAS.496.1922B; Penoyre et al. 2020,
https://ui.adsabs.harvard.edu/abs/2020MNRAS.495..321P). In this chapter we build a toy Gaia and watch this happen,
with every input known.

**You will learn**
1. where the photocentre of a binary is, and why equal-mass "twins" are invisible to astrometry;
2. how Gaia measures a star — one-dimensional along-scan positions — and how the five astrometric parameters are fitted
   by linear least squares;
3. what the unit-weight error (UWE) and the excess noise are, and how large they become for a binary of a given orbit;
4. how the signal depends on the orbital period compared with the length of the mission: noise, orbit, or curvature.

Everything here is simulated. Chapter 2 looks for the same effects in Gaia DR3, and Chapter 5 fits real Gaia DR4 epoch
astrometry with exactly the design matrix built below.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)                 # fixed seed: every number in this chapter is reproducible

## 1.1 The photocentre

Take two stars with masses $M_1 \ge M_2$, mass ratio $q = M_2/M_1$, and fluxes $F_1, F_2$ in Gaia's $G$ band, flux ratio
$\ell = F_2/F_1$. Let $\mathbf r = \mathbf r_2 - \mathbf r_1$ be the separation vector, whose length varies along a
Keplerian ellipse of semimajor axis $a$. Measured from the centre of mass, star 1 is at $-B\,\mathbf r$ and star 2 at
$(1-B)\,\mathbf r$, with $B = q/(1+q)$ the companion's fractional mass. The photocentre, the flux-weighted mean position,
is at $(\beta - B)\,\mathbf r$, with $\beta = \ell/(1+\ell)$ the companion's fractional flux. It therefore moves on an
ellipse of the same shape and period, with semimajor axis

$$ a_0 = a\,|B - \beta|, \qquad a_0\,[{\rm mas}] = \varpi\,[{\rm mas}]\;\frac{a}{\rm au}\;|B-\beta| , $$

where $\varpi$ is the parallax. Two limits matter: a **dark companion** ($\ell = 0$: a white dwarf, neutron star or
black hole) gives $a_0 = aB = a_1$, the orbit of the visible star itself; **twins** ($q = \ell = 1$) give $a_0 = 0$:
the light is balanced at the centre of mass, and astrometry sees nothing however wide the orbit.

For main-sequence companions we need $\ell(q)$. As a rough guide we use $\ell = q^4$, the steep mass–luminosity relation
of solar-type dwarfs in the optical; Chapter 3 replaces it with an isochrone.

In [ ]:
q = np.linspace(0.01, 1, 200)
B = q / (1 + q)                                  # fractional mass of the companion
for name, ell in [('dark companion ($\\ell = 0$)', 0 * q), ('main-sequence companion ($\\ell = q^4$)', q**4),
                  ('twin-like ($\\ell = q$)', q)]:
    beta = ell / (1 + ell)                       # fractional flux of the companion
    plt.plot(q, np.abs(B - beta), label=name)
plt.xlabel('mass ratio $q = M_2/M_1$'); plt.ylabel('$a_0 / a$')
plt.legend(); plt.xlim(0, 1); plt.ylim(0, 0.52); plt.show()

# identities (exact): a dark companion moves the photocentre on the primary's orbit, a twin does not move it at all
assert np.allclose(np.abs(B - q / (1 + q)), 0) and np.abs(B - beta)[-1] == 0   # beta of the last curve (ell = q) at q = 1

**Figure 1.1.** Photocentre semimajor axis $a_0$ in units of the relative semimajor axis $a$, against the mass ratio
$q$, for a dark companion (blue, $a_0/a = q/(1+q)$), a main-sequence companion with $\ell = q^4$ (orange), and a
companion as bright, relative to its mass, as the primary, $\ell = q$ (green, $a_0 = 0$ everywhere).

A dark companion produces the largest wobble at every $q$; this is why astrometry is so good at finding white dwarfs,
neutron stars and black holes around ordinary stars (Chapter 3). For main-sequence pairs the wobble peaks near
$q \approx 0.5$ and falls to zero for twins (orange). The "binary sequence" of equal-mass pairs that sits 0.75 mag
above the main sequence in a colour–magnitude diagram is therefore not where the wobble *per orbit* is largest
(Chapter 2 looks at what the data show for the whole population).

## 1.2 A toy Gaia

Gaia scans the sky continuously and measures each passing star in one dimension only: along the scan direction. Each
**transit** across the focal plane gives nine almost simultaneous measurements, one per CCD of the astrometric field,
all at the same scan angle $\theta$ (measured from north through east). The measured along-scan coordinate is

$$ w = \Delta\alpha_*\sin\theta + \Delta\delta\cos\theta , $$

where $(\Delta\alpha_*, \Delta\delta)$ is the star's displacement from a reference position ($\alpha_* = \alpha\cos\delta$).
For a single star the displacement at time $t$ (years from the reference epoch) is

$$ \Delta\alpha_* = \alpha_{*,0} + \mu_{\alpha*} t + \varpi f_\alpha(t), \qquad \Delta\delta = \delta_0 + \mu_\delta t +
\varpi f_\delta(t), $$

with $(f_\alpha, f_\delta)$ the parallax factors: the parallactic ellipse for unit parallax. So $w$ is **linear** in the five
astrometric parameters $\mathbf p = (\alpha_{*,0}, \delta_0, \varpi, \mu_{\alpha*}, \mu_\delta)$: $\mathbf w = \mathsf D\,\mathbf p$
with the design matrix $\mathsf D$ of rows $(\sin\theta, \cos\theta, f_{\rm AL}, t\sin\theta, t\cos\theta)$, where
$f_{\rm AL} = f_\alpha\sin\theta + f_\delta\cos\theta$. The fit is weighted linear least squares.

Our toy simplifies three things: transit times and scan angles are random (the real scanning law correlates them,
Chapter 5); the star sits at the ecliptic pole, so its parallactic ellipse is a unit circle; and the noise is Gaussian
with a known $\sigma$ per CCD. We use the DR3 baseline, 34 months, 30 transits, and $\sigma = 0.3$ mas, typical of
$G \approx 16$ (Lindegren et al. 2021, https://ui.adsabs.harvard.edu/abs/2021A&A...649A...2L).

In [ ]:
T_DR3 = 34 / 12            # DR3 baseline [yr] (25 July 2014 - 28 May 2017)
N_TR, N_CCD = 30, 9        # transits, and CCD measurements per transit
SIG = 0.3                  # along-scan uncertainty of one CCD measurement [mas]


def scans(T=T_DR3, n_tr=N_TR):
    """Random observation times t [yr, from the middle of the baseline] and scan angles theta [rad], one per CCD.

    The 9 CCDs of a transit share its time and scan angle (np.repeat copies each transit value 9 times)."""
    t = np.repeat(rng.uniform(-T / 2, T / 2, n_tr), N_CCD)
    theta = np.repeat(rng.uniform(0, 2 * np.pi, n_tr), N_CCD)
    return t, theta


def design(t, theta):
    """Design matrix (N, 5): w = D @ (alpha*_0, delta_0, parallax, pm_alpha*, pm_delta), all in mas and mas/yr.

    Parallax factors of a star at the ecliptic pole: a unit circle traced once a year."""
    s, c = np.sin(theta), np.cos(theta)
    f_al = np.cos(2 * np.pi * t) * s + np.sin(2 * np.pi * t) * c      # f_alpha sin(theta) + f_delta cos(theta)
    return np.column_stack([s, c, f_al, t * s, t * c])


def fit5(t, theta, w, sig=SIG):
    """Weighted least-squares single-star fit. Returns parameters, residuals and UWE = sqrt(chi^2 / (N - 5))."""
    D = design(t, theta)
    p, *_ = np.linalg.lstsq(D / sig, w / sig, rcond=None)      # dividing rows by sigma = weighting by 1/sigma^2
    r = w - D @ p
    return p, r, np.sqrt(np.sum((r / sig)**2) / (len(w) - 5))


# one single star: parallax 2 mas, proper motion (5, -3) mas/yr, observed with noise
p_true = np.array([0.0, 0.0, 2.0, 5.0, -3.0])
uwe1 = []
for k in range(300):
    t, th = scans()
    w = design(t, th) @ p_true + SIG * rng.normal(size=t.size)
    p, r, u = fit5(t, th, w)
    uwe1.append(u)
uwe1 = np.array(uwe1)
nu = N_TR * N_CCD - 5                                             # degrees of freedom
plt.hist(uwe1, 30, histtype='step', lw=2, density=True, label='300 simulated single stars')
x = np.linspace(0.8, 1.2, 200)
# if chi^2 follows a chi^2 distribution with nu degrees of freedom, UWE = sqrt(chi^2/nu) has density
# 2 x nu f_chi2(nu x^2) (change of variables)
from scipy import stats
plt.plot(x, 2 * x * nu * stats.chi2(nu).pdf(nu * x**2), 'k--', label=f'$\\chi^2$ with $\\nu = {nu}$')
plt.xlabel('UWE'); plt.ylabel('density'); plt.legend(); plt.show()
print(f'single stars: mean UWE^2 = {np.mean(uwe1**2):.4f} (expected 1, standard error {np.sqrt(2 / nu / 300):.4f})')
# calibration: the mean of chi^2/nu over 300 stars must be 1 within 3 standard errors
assert abs(np.mean(uwe1**2) - 1) < 3 * np.sqrt(2 / nu / 300)

**Figure 1.2.** Distribution of the unit-weight error of 300 simulated single stars fitted with the five-parameter
model (blue), and the distribution expected if $\chi^2$ follows the $\chi^2$ law with $\nu = 265$ degrees of freedom
(dashed).

For a correctly modelled star with correct uncertainties, $\chi^2/\nu$ scatters around 1 by $\sqrt{2/\nu} \approx 0.09$:
a UWE of 1.4 is more than four standard deviations out. Real Gaia uncertainties are not perfectly calibrated, which is
why DR3 publishes the **renormalised** UWE (RUWE): UWE divided by its typical value for stars of the same magnitude and
colour (Lindegren et al. 2021). We return to this in Chapter 2.

## 1.3 A short-period binary: the wobble becomes excess noise

Now add a photocentre orbit. With the Campbell elements — semimajor axis $a_0$, eccentricity $e$, inclination $i$,
argument of periastron $\omega$, longitude of the ascending node $\Omega$, period $P$ and time of periastron $T_p$ — the
photocentre displacement is

$$ \Delta\delta = A X + F Y, \qquad \Delta\alpha_* = B X + G Y, $$

with the elliptical rectangular coordinates $X = \cos E - e$, $Y = \sqrt{1-e^2}\sin E$ from the eccentric anomaly $E$
(Kepler's equation $E - e\sin E = 2\pi(t - T_p)/P$) and the **Thiele–Innes constants**

$$ A = a_0(\cos\omega\cos\Omega - \sin\omega\sin\Omega\cos i), \quad B = a_0(\cos\omega\sin\Omega + \sin\omega\cos\Omega\cos i), $$
$$ F = -a_0(\sin\omega\cos\Omega + \cos\omega\sin\Omega\cos i), \quad G = -a_0(\sin\omega\sin\Omega - \cos\omega\cos\Omega\cos i). $$

Gaia DR3 publishes orbits in exactly this form (Chapter 3). When the period is much shorter than the mission, the
wobble acts like extra random noise in each measurement. For a circular orbit, averaging $w^2$ over orbital phase and
over random scan angles gives the along-scan variance

$$ \sigma_{\rm orb}^2 = \frac{a_0^2}{4}\,(1 + \cos^2 i) , $$

so the rms is $a_0/\sqrt2$ for a face-on orbit, $a_0/2$ edge-on, and $a_0/\sqrt3$ in the mean over random orientations
(where $\langle\cos^2 i\rangle = 1/3$). Gaia measures it with the **excess noise** $\epsilon$: the extra noise that,
added in quadrature to each measurement's uncertainty, makes the fit statistically acceptable,
$\sum_k r_k^2/(\sigma_k^2 + \epsilon^2) = \nu$ (Lindegren et al. 2012, https://ui.adsabs.harvard.edu/abs/2012A&A...538A..78L).
Note that $\epsilon$ is an rms of the wobble projected on the scan direction, not $a_0$ itself.

One more factor. The nine CCD measurements of a transit are taken within seconds, so they share one value of the
wobble: the orbit is sampled only $N_{\rm tr} = 30$ times, not 270. The five fitted parameters can absorb part of any
signal; for a signal that is random from transit to transit they absorb on average a fraction $5/N_{\rm tr}$ of its
variance. The expected excess noise is therefore

$$ \epsilon \approx \frac{a_0}{2}\sqrt{1+\cos^2 i}\;\sqrt{1 - 5/N_{\rm tr}} . $$

In [ ]:
def kepler_xy(t, P, e, tp):
    """Elliptical rectangular coordinates X = cos E - e, Y = sqrt(1 - e^2) sin E at times t (same units as P, tp)."""
    M = 2 * np.pi * (t - tp) / P                 # mean anomaly
    E = M.copy()
    for _ in range(50):                          # Newton iterations for Kepler's equation E - e sin E = M
        E = E - (E - e * np.sin(E) - M) / (1 - e * np.cos(E))
    return np.cos(E) - e, np.sqrt(1 - e**2) * np.sin(E)


def thiele_innes(a0, i, omega, Omega):
    """Thiele-Innes constants A, B, F, G (units of a0) from Campbell elements (angles in radians)."""
    A = a0 * (np.cos(omega) * np.cos(Omega) - np.sin(omega) * np.sin(Omega) * np.cos(i))
    B = a0 * (np.cos(omega) * np.sin(Omega) + np.sin(omega) * np.cos(Omega) * np.cos(i))
    F = -a0 * (np.sin(omega) * np.cos(Omega) + np.cos(omega) * np.sin(Omega) * np.cos(i))
    G = -a0 * (np.sin(omega) * np.sin(Omega) - np.cos(omega) * np.cos(Omega) * np.cos(i))
    return A, B, F, G


def wobble_al(t, theta, a0, P, e, i, omega, Omega, tp):
    """Along-scan displacement [units of a0] of the photocentre orbit."""
    A, B, F, G = thiele_innes(a0, i, omega, Omega)
    X, Y = kepler_xy(t, P, e, tp)
    return (B * X + G * Y) * np.sin(theta) + (A * X + F * Y) * np.cos(theta)


def excess_noise(r, sig=SIG, nu=None):
    """Excess noise eps [mas]: sum r^2 / (sig^2 + eps^2) = nu (0 if the fit is already acceptable). Solved by bisection."""
    nu = nu or len(r) - 5
    if np.sum((r / sig)**2) <= nu:
        return 0.0
    lo, hi = 0.0, 10 * np.std(r)
    for _ in range(60):
        mid = (lo + hi) / 2
        lo, hi = (mid, hi) if np.sum(r**2 / (sig**2 + mid**2)) > nu else (lo, mid)
    return (lo + hi) / 2


def observe(a0, P, e=0.0, i=0.0, T=T_DR3, n_tr=N_TR, sig=SIG):
    """Simulate one binary (random omega, Omega, periastron time) and fit the single-star model.

    Returns (UWE, excess noise [mas], fitted parameters)."""
    t, th = scans(T, n_tr)
    w = design(t, th) @ p_true + wobble_al(t, th, a0, P, e, i, *rng.uniform(0, 2 * np.pi, 2), rng.uniform(0, P))
    w += sig * rng.normal(size=t.size)
    p, r, u = fit5(t, th, w, sig)
    return u, excess_noise(r, sig), p


a0 = 1.0                                         # photocentre semimajor axis [mas]
inc = np.radians([0, 30, 45, 60, 75, 90])
eps_ratio = np.array([[observe(a0, P=0.1, i=i)[1] / a0 for _ in range(200)] for i in inc])   # (6, 200)
cosi = np.linspace(0, 1, 100)
plt.plot(cosi, np.sqrt(1 + cosi**2) / 2, 'k:', label='$\\sqrt{1+\\cos^2 i}\\,/\\,2$')
plt.plot(cosi, np.sqrt(1 + cosi**2) / 2 * np.sqrt(1 - 5 / N_TR), 'k-', label='$\\times\\sqrt{1 - 5/N_{\\rm tr}}$')
plt.errorbar(np.cos(inc), eps_ratio.mean(1), eps_ratio.std(1), fmt='o', label='toy Gaia, P = 0.1 yr (mean and scatter of 200)')
plt.xlabel('$\\cos i$'); plt.ylabel('excess noise $\\epsilon$ / $a_0$'); plt.legend(); plt.show()
pred = np.sqrt(1 + np.cos(inc)**2) / 2 * np.sqrt(1 - 5 / N_TR)      # with the absorbed fraction 5 / N_tr
print('epsilon / a0, simulated:', np.round(eps_ratio.mean(1), 3), '\n              predicted:', np.round(pred, 3))
# calibration: the mean excess noise follows the analytic projection within 3% at every inclination
assert np.all(np.abs(eps_ratio.mean(1) / pred - 1) < 0.03)

**Figure 1.3.** Excess noise $\epsilon$ of the single-star fit, in units of the photocentre semimajor axis $a_0 = 1$ mas,
for circular orbits with $P = 0.1$ yr, against $\cos i$ (face-on at the right). Points: mean and standard deviation of
200 simulated binaries per inclination (random $\omega$, $\Omega$, phase and scan angles; $\sigma = 0.3$ mas per CCD).
Dotted line: the along-scan rms of the orbit, $a_0\sqrt{1+\cos^2 i}/2$; solid line: the same times
$\sqrt{1 - 5/N_{\rm tr}}$, the part not absorbed by the five astrometric parameters.

The simulated excess noise follows the prediction (solid line): the wobble has become extra noise, and its size is
the along-scan rms of the orbit, between $a_0/2$ (edge-on) and $a_0/\sqrt 2$ (face-on), reduced by 9% because the fit
absorbs part of a signal sampled only 30 times (the dotted line ignores this, and misses every point). Two lessons. First, the excess
noise is not $a_0$: converting one to the other needs an orientation, which we do not know for an unresolved binary.
Second, the scatter (error bars) is large, ±20%: with 30 transits a single star's excess noise is a noisy estimate
of its wobble. In real Gaia data the excess noise also contains the calibration and attitude errors that our toy
lacks; for bright stars these set a floor of ~0.1 mas (Chapter 2).

## 1.4 Noise, orbit or curvature: the role of the period

How does the misfit depend on the period $P$ compared with the baseline $T$? We look at two families of binaries.

- **Fixed $a_0 = 1$ mas**, any period: this isolates the time sampling.
- **Fixed masses and distance**: a $1\,M_\odot$ star with a dark $0.5\,M_\odot$ companion at 1 kpc ($\varpi = 1$ mas).
  Kepler's third law, $a^3\,[{\rm au}] = (M_1+M_2)\,[M_\odot]\;P^2\,[{\rm yr}]$, makes $a_0 \propto P^{2/3}$: longer
  periods mean wider orbits.

For the second family we also compute the expected $\chi^2$ exactly. If the data are $\mathbf w = \mathsf D\mathbf p +
\mathbf s + \mathbf n$, with orbital signal $\mathbf s$ and noise $\mathbf n$, least squares removes from $\mathbf s$ the
part that looks like a change of position, parallax or proper motion: in units of $\sigma$, the residual is
$(\mathsf I - \mathsf H)(\mathbf s + \mathbf n)$, with the projection ("hat") matrix
$\mathsf H = \mathsf D(\mathsf D^{\rm T}\mathsf D)^{-1}\mathsf D^{\rm T}$. Hence

$$ \mathbb E[\chi^2] = \nu + \lVert(\mathsf I - \mathsf H)\,\mathbf s/\sigma\rVert^2 . $$

The signal that the five parameters absorb is invisible in $\chi^2$, and that is what happens to long periods.

In [ ]:
periods = np.logspace(-1.5, 1.7, 25)            # 0.03 to 50 yr
M1, M2, plx = 1.0, 0.5, 1.0                      # [Msun], [Msun], [mas]


def a0_of_P(P):
    """Photocentre semimajor axis [mas] of the fixed-mass binary (dark companion: a0 = a * M2 / (M1 + M2))."""
    return plx * ((M1 + M2) * P**2)**(1 / 3) * M2 / (M1 + M2)


uwe_fix_a0 = np.array([np.median([observe(1.0, P, i=np.arccos(rng.uniform()))[0] for _ in range(100)]) for P in periods])
uwe_fix_m = np.array([np.median([observe(a0_of_P(P), P, i=np.arccos(rng.uniform()))[0] for _ in range(100)]) for P in periods])
uwe_fix_m66 = np.array([np.median([observe(a0_of_P(P), P, i=np.arccos(rng.uniform()), T=66 / 12, n_tr=55)[0]
                                   for _ in range(100)]) for P in periods])

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].semilogx(periods, uwe_fix_a0, 'o-')
ax[0].set(title='fixed $a_0$ = 1 mas', xlabel='period P [yr]', ylabel='median UWE')
ax[1].semilogx(periods, uwe_fix_m, 'o-', label='DR3: 34 months, 30 transits')
ax[1].semilogx(periods, uwe_fix_m66, 's-', label='DR4: 66 months, 55 transits')
ax[1].set(title='fixed masses: 1 + 0.5 $M_\\odot$ (dark) at 1 kpc', xlabel='period P [yr]')
for a in ax:
    a.axvline(T_DR3, color='k', ls=':', lw=1); a.axhline(1.4, color='r', ls='--', lw=1)
ax[1].axvline(66 / 12, color='C1', ls=':', lw=1); ax[1].legend()
plt.tight_layout(); plt.show()

P_peak = periods[np.argmax(uwe_fix_m)]
print(f'fixed masses: median UWE peaks at P = {P_peak:.1f} yr (DR3 baseline {T_DR3:.1f} yr)')
# the peak of the fixed-mass curve lies within a factor of 2 of the baseline
assert 0.5 * T_DR3 < P_peak < 2 * T_DR3

**Figure 1.4.** Median UWE of 100 simulated binaries per period (random orientations and phases), against the orbital
period. Left: fixed photocentre semimajor axis $a_0 = 1$ mas. Right: fixed masses, $1\,M_\odot$ with a dark
$0.5\,M_\odot$ companion at 1 kpc, so that $a_0 \propto P^{2/3}$, for the DR3 (blue) and DR4 (orange) baselines.
Dotted vertical lines: the DR3 (34 months) and DR4 (66 months) baselines; dashed red line: UWE = 1.4.

At fixed $a_0$ (left) the UWE is flat for $P \ll T$ — the regime of Section 1.3 — and drops for $P \gtrsim T$: Gaia sees
only part of the orbit, and over a short arc the orbit is nearly a straight line, which the proper motion absorbs. At
fixed masses (right) two effects compete: $a_0$ grows as $P^{2/3}$, while absorption grows with $P$. The product
peaks near the baseline. For this binary ($a_0 = 0.38$ mas at $P = 1$ yr, comparable to the 0.3-mas noise of one CCD
measurement) the DR3 median UWE exceeds 1.4 only for periods of about 2–3 yr. The DR4 baseline (orange) almost doubles
the number of transits and the time span: the peak is higher and moves to $P \approx 6$ yr, and the median UWE is above
1.4 from about 2 to 10 yr. Short-period orbits gain little, because their tiny wobble stays below the noise. Binaries
that were too slow for DR3 become detectable in DR4, which is what happened to Gaia BH3 (Chapter 5).

Where does the absorbed signal go? Into the proper motion and the parallax. A star whose orbit is much longer than
the mission has a single-star proper motion that is the instantaneous orbital velocity of the photocentre plus the
systemic motion; comparing it with a long-baseline proper motion (Hipparcos–Gaia) reveals the companion
(Kervella et al. 2019, https://ui.adsabs.harvard.edu/abs/2019A&A...623A..72K; Brandt 2021,
https://ui.adsabs.harvard.edu/abs/2021ApJS..254...42B). The exact $\chi^2$ formula makes the absorption quantitative:

In [ ]:
# one fixed binary and one fixed set of scans; 2000 noise realisations
t, th = scans()
D = design(t, th)
H = D @ np.linalg.solve(D.T @ D, D.T)                       # hat matrix (N, N): projection onto the 5 parameters
res = []
for P in (0.3, T_DR3, 15.0):
    s = wobble_al(t, th, a0_of_P(P), P, 0.3, np.radians(60), 1.0, 2.0, 0.1 * P)    # e = 0.3, i = 60 deg
    chi2 = [fit5(t, th, D @ p_true + s + SIG * rng.normal(size=t.size))[2]**2 * nu for _ in range(2000)]
    expect = nu + np.sum(((s - H @ s) / SIG)**2)
    absorbed = 1 - np.sum((s - H @ s)**2) / np.sum((s - s.mean())**2)
    res.append((P, np.mean(chi2), expect, np.std(chi2) / np.sqrt(2000), absorbed))
    print(f'P = {P:5.2f} yr: <chi^2> = {np.mean(chi2):9.1f}, nu + |(I-H)s|^2 = {expect:9.1f}; '
          f'fraction of the signal variance absorbed by the 5 parameters: {absorbed:.2f}')
# identity (statistical): the ensemble mean of chi^2 equals the projection formula within 3 standard errors
assert all(abs(m - e) < 3 * se for P, m, e, se, f in res)

The ensemble mean of $\chi^2$ matches the projection formula, and the absorbed fraction makes the trend of Figure 1.4
explicit. For this one set of 30 scans, the five parameters absorb 28% of the signal variance at $P = 0.3$ yr (the
average over scan sets is $5/30 = 17\%$), half at $P \approx T$, and 99% at $P = 15$ yr, where the binary is
practically invisible in $\chi^2$.

## Summary

- The photocentre of a binary moves on an ellipse of semimajor axis $a_0 = a\,|B - \beta|$: a dark companion gives the
  largest wobble, equal-mass twins none.
- Gaia measures one-dimensional along-scan positions; the single-star model is linear in five parameters and is
  fitted by least squares. UWE $= \sqrt{\chi^2/\nu}$ is 1 for a well-modelled star.
- For $P \ll T$ the wobble becomes excess noise with rms $a_0\sqrt{1+\cos^2 i}/2$; for $P \gg T$ it is absorbed into
  the proper motion. At fixed masses the UWE peaks near $P \approx T$, and a longer baseline extends the reach.

**Exercises.** (1) Repeat Figure 1.3 with $e = 0.7$: how does the excess noise change? (2) In Figure 1.4, replace the
dark companion with a main-sequence one ($\ell = q^4$, $q = 0.9$). (3) For $P = 15$ yr, compare the fitted proper motion
with the true one: how large is the error, in units of its formal uncertainty?